# Packages import

In [4]:
import os
import json
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# Ceneo Scraper

0. Prepare selenium


1. Provide url of the product's opinions page

In [4]:
product_code = "179676915"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-(page)"

2. Send request to provided url

In [3]:
path_to_driver = "D:\\chromedriver-win64\\chromedriver.exe"
s = Service(path_to_driver)
driver = webdriver.Chrome(service=s)
driver.get(url)
driver.maximize_window()
driver.find_element(by = 'xpath', value="//*[@id='js_cookie-consent-general']/div/div[2]/button[1]").click()

NameError: name 'Service' is not defined

In [5]:
response = requests.get(url)
print(response.status_code)


200


3. Fetch product name

In [6]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [7]:
product_name = page_dom.select_one("h1").get_text()
print(product_name)
print(type(product_name))

Redmi Note 14 Pro 4G 12/512GB Czarny
<class 'str'>


4. Fetch all opinions from the webpage

In [8]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


5. Parse opinions to extract required data

In [32]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion.get("data-entry-id"),
        'author' : opinion.select_one("span.user-post__author-name").get_text().strip(),
        'recommendation' : opinion.select_one("span.user-post__author-recomendation > em").get_text() if opinion.select_one("span.user-post__author-recomendation > em") else None ,
        'score' : opinion.select_one("span.user-post__score-count").get_text().strip(),
        'content' : opinion.select_one("div.user-post__text").get_text().strip(),
        'pros' : [p.get_text().strip() for p in opinion.select("div.review-feature__item--positive")],
        'cons' : [c.get_text().strip() for c in opinion.select("div.review-feature__item--negative")],
        'helpful' : opinion.select_one("button.vote-yes > span").get_text().strip(),
        'unhelpful' : opinion.select_one("button.vote-no > span").get_text().strip(),
        'publish_date' : opinion.select_one("span.user-post__published > time:nth-child(1)").get("datetime").strip(),
        'purchase_date' : opinion.select_one("span.user-post__published > time:nth-child(2)").get("datetime").strip() if opinion.select_one("span.user-post__published > time:nth-child(2)") else None ,
    }
    all_opinions.append(single_opinion)

6. Check if there is net page with opinons


In [2]:
driver.find_element(by = 'xpath', value='//*[@id="reviews"]/div/div[6]/button[4]').click()

NameError: name 'driver' is not defined

In [20]:
next = True if  page_dom.select_one('button.pagiation__next') else False
if next: page += 1 

8. Save acquired opinions

In [ ]:
if not os.path.exists("./opinions"):
    os.mkdir('./opinions')

In [33]:
with open(f'./{product_code}.json', 'w', encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent= 4, ensure_ascii=False)